<a href="https://colab.research.google.com/github/Gabriel-Roledo-ds/indicadores_de_inova-o_tecnologica/blob/main/indicadores_de_inova%C3%A7%C3%A3o_tecnologica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise de Indicadores de Inovação Tecnológica
### Trabalho de Economia da Informação — Ciência de Dados - *FATEC* Ourinhos
Metodologia: CRISP-DM

## Sobre este notebook

Este notebook segue a metodologia **CRISP-DM** (Cross-Industry Standard Process for Data Mining),
um roteiro padrão de mercado para projetos de dados, dividido em 6 etapas:

1. **Entendimento do Negócio** — que perguntas queremos responder e por quê
2. **Entendimento dos Dados** — quais bases temos, o que elas contêm, como estão estruturadas
3. **Preparação dos Dados** — limpeza, padronização e consolidação das bases
4. **Modelagem** — construção dos rankings e análise de correlação entre indicadores
5. **Avaliação** — os resultados fazem sentido? respondem às perguntas da Etapa 1?
6. **Implantação** — exportação das tabelas finais e conclusões do relatório

>  Na prática, essas etapas não são 100% lineares - é comum voltar a uma etapa anterior
> ao descobrir algo novo (ex: um problema nos dados só aparece na Preparação, mesmo já
> tendo passado pela Etapa 2). Isso é normal e faz parte do processo.

# 1. Entendimento do Negócio

**Objetivo:** avaliar o nível de inovação tecnológica de diferentes países, comparando
esforço (investimento) com resultado (produção de conhecimento/tecnologia).

**Perguntas que este notebook busca responder:**
- Quais países mais investem em P&D e possuem mais pesquisadores?
- Quais países mais geram patentes, marcas, desenhos industriais e exportações de alta tecnologia?
- Existe relação entre esforço e resultado?
- Como o Brasil se posiciona frente aos líderes?
- Como isso evoluiu nos últimos 5 recortes de 5 em 5 anos?

**Critério de sucesso:** base consolidada e confiável, com rankings Top 10 por indicador/ano,
sustentando as tabelas da Fase 1 (22/09/2026) e as conclusões da Fase 2 (09/11/2026).

# 2. Entendimento dos Dados

Fontes utilizadas:
- **[Gasto em P&D (% do PIB)](https://data.worldbank.org/indicator/GB.XPD.RSDV.GD.ZS )** — Banco Mundial
- **[Pesquisadores em P&D](https://data.worldbank.org/indicator/SP.POP.SCIE.RD.P6 )** — Banco Mundial
- **[Pedidos de patentes](https://data.worldbank.org/indicator/IP.PAT.RESD)** — Banco Mundial
- **[Exportações de alta tecnologia](https://data.worldbank.org/indicator/TX.VAL.TECH.MF.ZS)** — Banco Mundial
- **[Patentes, Marcas e Desenhos Industriais](https://www3.wipo.int/ipstats/ips-search/countryprofiles)** — WIPO

Os 4 indicadores do Banco Mundial vêm em formato *wide* (anos em colunas), com metadados
de país e indicador em arquivos separados. A base da WIPO tem formato ainda a ser explorado.

Para facilitar o entendimento, os arquivos foram renomeados com titulos descritivos para esse notebook.

##Setup e Imports

In [43]:

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
pd.set_option('display.float_format', '{:.2f}'.format)

CAMINHO = "/content/drive/MyDrive/fatec/Indicadores_de_inovacao_tec/"

anos_bm = [2021, 2016, 2011, 2006, 2001]
anos_wipo = [2024, 2019, 2014, 2009, 2004]

#Estabeleço a conexão com o drive
#Faço o import da biblioteca para manipular a base
#Defino variáveis com os anos da frequência 5 (Tabela com frequências no guia do trabalho)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


###Funções

In [44]:


def carregar_indicador_bm(caminho_dados, caminho_metadata_country, nome_indicador, anos):
    """Carrega um indicador do Banco Mundial em formato long, sem agregados regionais."""
    df = pd.read_csv(caminho_dados, skiprows=4)

    meta = pd.read_csv(caminho_metadata_country)
    paises_validos = meta.loc[meta["Region"].notna(), "Country Code"]
    df = df[df["Country Code"].isin(paises_validos)]

    df_long = df.melt(
        id_vars=["Country Name", "Country Code"],
        value_vars=[str(a) for a in anos],
        var_name="Ano",
        value_name="Valor"
    )
    df_long["Ano"] = df_long["Ano"].astype(int)
    df_long["Indicador"] = nome_indicador
    return df_long

#____________________________________________________________________


def diagnostico(df):
    """Tabela resumo: tipo, % de nulos e valores únicos por coluna."""
    return pd.DataFrame({
        "Tipo": df.dtypes,
        "Nulos (%)": (df.isna().mean() * 100).round(2),
        "Valores únicos": df.nunique()
    })

#____________________________________________________________________

def diagnostico_completo(df, coluna_valor="Valor", coluna_ano="Ano"):
    """Checklist de confiabilidade — rodar só após o dado estar em formato long."""
    print("== Tipos e nulos ==")
    display(diagnostico(df))

    print("\n== Estatísticas descritivas ==")
    display(df[coluna_valor].describe())

    print("\n== Cobertura por ano (nº de países com dado) ==")
    display(df.groupby(coluna_ano)[coluna_valor].count())

    print("\n== Duplicatas país+ano ==")
    print(df.duplicated(subset=["Country Code", coluna_ano]).sum())

#______________________________________________________________________


def carregar_indicador_wipo(caminho_dados, nome_indicador_wipo, nome_indicador, anos):
    """
    Carrega um indicador da base WIPO (patentes/marcas/desenhos) em formato long.
    nome_indicador_wipo: texto exato da coluna 'Statistics' que identifica o indicador desejado
    nome_indicador: rótulo que usaremos no dataframe consolidado
    """
    df = pd.read_csv(caminho_dados, skiprows=6, index_col=False)

    df = df[df["Statistics"] == nome_indicador_wipo]

    df_long = df.melt(
        id_vars=["Origin", "Origin (Code)"],
        value_vars=[str(a) for a in anos],
        var_name="Ano",
        value_name="Valor"
    )
    df_long["Ano"] = df_long["Ano"].astype(int)
    df_long["Indicador"] = nome_indicador
    df_long = df_long.rename(columns={"Origin": "Country Name", "Origin (Code)": "Country Code"})

    return df_long

#_______________________________________________________________________________

def top10_por_indicador_ano(df, indicador, ano):
    """Retorna o Top 10 países de um indicador, em um ano específico, ordenado do maior para o menor valor."""
    filtro = (df["Indicador"] == indicador) & (df["Ano"] == ano)
    return (
        df[filtro]
        .dropna(subset=["Valor"])
        .sort_values("Valor", ascending=False)
        .head(10)
        [["Country Name", "Valor"]]
        .reset_index(drop=True)
    )

##Exploração inicial: Gasto em P&D (% do PIB), cru

In [45]:

df_teste = pd.read_csv(
    CAMINHO + "Indicadores_PeD/pesquisa_e_desenvolvimento_em_proporção_ao_PIB/pesquisa_e_desenvolvimento_em_proporção_ao_PIB.csv",
    skiprows=4  # pula os metadados soltos antes da tabela real
)


df_teste.shape

(265, 71)

In [46]:
df_teste.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,Research and development expenditure (% of GDP),GB.XPD.RSDV.GD.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,Research and development expenditure (% of GDP),GB.XPD.RSDV.GD.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,Research and development expenditure (% of GDP),GB.XPD.RSDV.GD.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Africa Western and Central,AFW,Research and development expenditure (% of GDP),GB.XPD.RSDV.GD.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Angola,AGO,Research and development expenditure (% of GDP),GB.XPD.RSDV.GD.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



*   Formato wide — cada ano é uma coluna, cada país é uma linha. Será feito um `.melt` para transformar as colunas de ano em duas colunas: `Ano` e `Valor`. O formato long (Country Name / Country Code / Ano / Valor) vai facilitar as futuras comparações entre tabelas.

*   Existem "países" que na verdade são agregados regionais/blocos de renda (World, OECD members...), que precisam ser removidos antes do ranking.

*   Foi necessário o uso de `skiprows` porque o pandas lia o título no lugar do cabeçalho da tabela.

##Exploração inicial: WIPO (Patentes, Marcas, Desenhos Industriais), cru

In [47]:
df_wipo_bruto = pd.read_csv(
    CAMINHO + "Indicadores_PeD/patents_trademarks_industrial_design.csv",
    skiprows=6,
    index_col=False
)

df_wipo_bruto.shape



(522, 25)

In [48]:
df_wipo_bruto.columns.tolist()

['Origin',
 'Origin (Code)',
 'Office',
 'Statistics',
 '2004',
 '2005',
 '2006',
 '2007',
 '2008',
 '2009',
 '2010',
 '2011',
 '2012',
 '2013',
 '2014',
 '2015',
 '2016',
 '2017',
 '2018',
 '2019',
 '2020',
 '2021',
 '2022',
 '2023',
 '2024']

In [49]:
df_wipo_bruto.head(5)

,Origin,Origin (Code),Office,Statistics,2004,2005,2006,2007,2008,2009,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Albania,AL,Total,1.1 - Patent - Total patent applications,NaN,1.00,NaN,NaN,NaN,4.00,...,16.00,37.00,18.00,21.00,12.00,NaN,32.00,28.00,28.00,NaN
1,Albania,AL,Total,2.1 - Trademark - Total classes in trademark a...,NaN,NaN,NaN,NaN,NaN,NaN,...,966.00,1019.00,1310.00,1775.00,1527.00,1630.00,2132.00,1523.00,2155.00,2440.00
2,Albania,AL,Total,3.1 - Industrial design - Total designs in app...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,58.00,58.00,50.00,234.00,31.00
3,Algeria,DZ,Total,1.1 - Patent - Total patent applications,59.00,65.00,59.00,88.00,NaN,NaN,...,107.00,112.00,158.00,162.00,119.00,170.00,280.00,483.00,1412.00,1117.00
4,Algeria,DZ,Total,2.1 - Trademark - Total classes in trademark a...,NaN,NaN,NaN,NaN,NaN,NaN,...,14638.00,NaN,8580.00,7350.00,9816.00,12561.00,11445.00,11868.00,14314.00,16033.00


###Identificar coluna de indicador na base WIPO

In [50]:
for col in df_wipo_bruto.columns:
    n_unicos = df_wipo_bruto[col].nunique()
    if n_unicos <= 15:
        print(f"{col}: {n_unicos} valores únicos -> {df_wipo_bruto[col].unique()}")

Office: 1 valores únicos -> ['Total']
Statistics: 3 valores únicos -> ['1.1 - Patent - Total patent applications'
 '2.1 - Trademark - Total classes in trademark applications'
 '3.1 - Industrial design - Total designs in applications']


In [51]:
print("Total de origens únicas:", df_wipo_bruto["Origin"].nunique())
print(sorted(df_wipo_bruto["Origin"].unique()))

Total de origens únicas: 187
['Albania', 'Algeria', 'Andorra', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bhutan', 'Bolivia (Plurinational State of)', 'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cambodia', 'Cameroon', 'Canada', 'Central African Republic', 'Chad', 'Chile', 'China', 'China, Hong Kong SAR', 'China, Macao SAR', 'Colombia', 'Comoros', 'Congo', 'Costa Rica', 'Croatia', 'Cuba', 'Curaçao', 'Cyprus', 'Czech Republic', "Côte d'Ivoire", "Democratic People's Republic of Korea", 'Democratic Republic of the Congo', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador', 'Equatorial Guinea', 'Estonia', 'Eswatini', 'Ethiopia', 'Finland', 'France', 'Gabon', 'Gambia', 'Georgia', 'Germany', 'Ghana', 'Greece', 'Grenada', 'Guatemala', 'G

## Observações base WIPO

### Estrutura da base WIPO
- Coluna que identifica o indicador: `Statistics`
- Valores encontrados: `1.1 - Patent - Total patent applications`, `2.1 - Trademark - Total classes in trademark applications`, `3.1 - Industrial design - Total designs in applications`
- Formato da base: wide (anos 2004-2024 em colunas), com os 3 indicadores empilhados como linhas (3 linhas por país) — diferente do Banco Mundial, que tem 1 arquivo por indicador
- Coluna de país: `Origin` (nome) / `Origin (Code)` (código ISO2, ex: "AL") — coluna `Office` é sempre "Total", não carrega informação útil aqui

###Divergência de nomes de país entre fontes


A base WIPO (187 origens, sem agregados regionais) usa nomenclatura diferente do
Banco Mundial para vários países. Exemplos:

| Banco Mundial | WIPO |
|---|---|
| United States | United States of America |
| Korea, Rep. | Republic of Korea |
| Iran, Islamic Rep. | Iran (Islamic Republic of) |
| Egypt, Arab Rep. | Egypt |
| Venezuela, RB | Venezuela (Bolivarian Republic of) |
| Czechia | Czech Republic |
| Bahamas, The | Bahamas |

**Implicação para a consolidação:** um `merge` direto por nome de país vai falhar
silenciosamente nesses casos (linhas não vão casar). Duas opções a avaliar na Etapa 3:
- Usar o **código de país (ISO)** como chave de junção, quando disponível e no mesmo padrão nas duas fontes
- Construir um dicionário manual de correspondência de nomes divergentes

#3. Preparação dos Dados

Nesta etapa:
- Removemos agregados regionais/blocos de renda (não são países)
- Convertemos os 4 indicadores do Banco Mundial de *wide* para *long*
- Filtramos apenas os 5 anos de referência
- Adequamos a base WIPO ao mesmo formato
- Rodamos diagnóstico de confiabilidade em cada indicador antes de consolidar

##Carregamento e limpeza: Gasto em P&D (% do PIB)

In [52]:

df_pd_pib = carregar_indicador_bm(
    CAMINHO + "Indicadores_PeD/pesquisa_e_desenvolvimento_em_proporção_ao_PIB/pesquisa_e_desenvolvimento_em_proporção_ao_PIB.csv",
    CAMINHO + "Indicadores_PeD/pesquisa_e_desenvolvimento_em_proporção_ao_PIB/metadata_country_pesquisa_e_desenvolvimento_em_proporção_ao_PIB.csv",
    "Gasto em P&D (% do PIB)",
    anos_bm
)

df_pd_pib.shape
df_pd_pib["Country Name"].nunique()

217

In [53]:
df_pd_pib.head()

,Country Name,Country Code,Ano,Valor,Indicador
0,Aruba,ABW,2021,NaN,Gasto em P&D (% do PIB)
1,Afghanistan,AFG,2021,NaN,Gasto em P&D (% do PIB)
2,Angola,AGO,2021,NaN,Gasto em P&D (% do PIB)
3,Albania,ALB,2021,0.19,Gasto em P&D (% do PIB)
4,Andorra,AND,2021,NaN,Gasto em P&D (% do PIB)


##Carregamento e limpeza: Pesquisadores em P&D

In [54]:

df_pesquisadores = carregar_indicador_bm(
    CAMINHO + "Indicadores_PeD/Pesquisadores envolvidos em P&D/pesquisadores_envolvidos_em_P&D.csv",
    CAMINHO + "Indicadores_PeD/Pesquisadores envolvidos em P&D/metadata_country_pesquisadores_envolvidos_em_P&D.csv",
    "Pesquisadores em P&D (por milhão hab.)",
    anos_bm
)

df_pesquisadores.shape
df_pesquisadores["Country Name"].nunique()

217

In [55]:
df_pesquisadores.head()

,Country Name,Country Code,Ano,Valor,Indicador
0,Aruba,ABW,2021,NaN,Pesquisadores em P&D (por milhão hab.)
1,Afghanistan,AFG,2021,NaN,Pesquisadores em P&D (por milhão hab.)
2,Angola,AGO,2021,NaN,Pesquisadores em P&D (por milhão hab.)
3,Albania,ALB,2021,449.10,Pesquisadores em P&D (por milhão hab.)
4,Andorra,AND,2021,NaN,Pesquisadores em P&D (por milhão hab.)


##Carregamento e limpeza: Pedidos de patentes

In [56]:

df_patentes_resd = carregar_indicador_bm(
    CAMINHO + "Indicadores_PeD/pedidos_de_patente/pedidos_de_patente.csv",
    CAMINHO + "Indicadores_PeD/pedidos_de_patente/metadata_country_pedidos_de_patente.csv",
    "Pedidos de patentes",
    anos_bm
)

df_patentes_resd.shape
df_patentes_resd["Country Name"].nunique()

217

In [57]:
df_patentes_resd.head()

,Country Name,Country Code,Ano,Valor,Indicador
0,Aruba,ABW,2021,NaN,Pedidos de patentes
1,Afghanistan,AFG,2021,NaN,Pedidos de patentes
2,Angola,AGO,2021,NaN,Pedidos de patentes
3,Albania,ALB,2021,23.00,Pedidos de patentes
4,Andorra,AND,2021,3.00,Pedidos de patentes


##Carregamento e limpeza: Exportações de alta tecnologia

In [58]:

df_exportacoes = carregar_indicador_bm(
    CAMINHO + "Indicadores_PeD/exportações de alta tecnologia/exportações_de_alta_tecnologia.csv",
    CAMINHO + "Indicadores_PeD/exportações de alta tecnologia/metadata_exportações_de_alta tecnologia.csv",
    "Exportações de alta tecnologia (%)",
    anos_bm
)

df_exportacoes.shape
df_exportacoes["Country Name"].nunique()

217

In [59]:
df_exportacoes.head()

,Country Name,Country Code,Ano,Valor,Indicador
0,Aruba,ABW,2021,4.82,Exportações de alta tecnologia (%)
1,Afghanistan,AFG,2021,NaN,Exportações de alta tecnologia (%)
2,Angola,AGO,2021,16.13,Exportações de alta tecnologia (%)
3,Albania,ALB,2021,0.42,Exportações de alta tecnologia (%)
4,Andorra,AND,2021,24.38,Exportações de alta tecnologia (%)


##Diagnóstico: Gasto em P&D (% do PIB)

In [60]:
diagnostico_completo(df_pd_pib)

== Tipos e nulos ==


,Tipo,Nulos (%),Valores únicos
Country Name,object,0.00,217
Country Code,object,0.00,217
Ano,int64,0.00,5
Valor,float64,59.63,438
Indicador,object,0.00,1



== Estatísticas descritivas ==


,Valor
count,438.00
mean,0.99
std,1.01
min,0.01
25%,0.23
50%,0.62
75%,1.43
max,5.76



== Cobertura por ano (nº de países com dado) ==


,Valor
Ano,
2001,82
2006,85
2011,92
2016,91
2021,88



== Duplicatas país+ano ==
0


**Estatística:**
___

**count 438**

Quantidade de valores não nulos usados no cálculo. Você tem 5 anos × 217 países = 1085 combinações possíveis, mas só 438 têm dado de verdade (o resto é NaN) — bate com os 59,63% de nulos que vimos no diagnóstico de tipos.
___

**mean 0.99**

A média — soma de todos os 438 valores, dividida por 438. Em média, os países da amostra gastam 0,99% do PIB em P&D.
___
**std 1.01 (desvio padrão)**

Mede o quanto os valores variam em torno da média. Um desvio padrão de 1,01, quase do tamanho da própria média (0,99), indica que os dados são bem dispersos — tem muita diferença entre os países, não é um grupo homogêneo. Isso já é esperado: países como Israel/Coreia do Sul gastam bem mais que 3% do PIB, enquanto muitos países em desenvolvimento gastam frações de 1%.
___
**min 0.0126**

O menor valor de gasto em P&D encontrado na base inteira (entre todos os 438 registros) — 0,0126% do PIB, praticamente nada.
___
**max 5.76**

O maior valor encontrado — um país (não sabemos qual ainda, sem olhar) chegou a investir quase 6% do PIB em P&D num dos 5 anos-base. Isso é um valor alto mas plausível (Israel e Coreia do Sul historicamente ficam nessa faixa), então não é um outlier suspeito de erro.
___

**25%, 50%, 75% (quartis)**


   Esses três dividem os 438 valores em 4 partes iguais, depois de ordenados do menor para o maior

* 25% = 0.233 → 25% dos registros têm gasto abaixo de 0,233% do PIB (ou seja, um quarto dos países-ano são bem pouco investidores)

* 50% = 0.617 (a mediana) → metade dos registros está abaixo desse valor, metade acima. Repare que a mediana (0,617) é bem menor que a média (0,99) — isso é um sinal importante: significa que a distribuição é assimétrica, puxada para cima por poucos países que investem muito (tipo Israel, Coreia do Sul), enquanto a maioria dos países fica bem abaixo da média.

* 75% = 1.43 → 75% dos registros estão abaixo de 1,43%; só o quarto "de cima" (os 25% mais investidores) ultrapassa esse valor.

___

**Padrão ao longo do tempo:** a cobertura sobe de 2001 (82) até 2011 (92), e depois cai levemente até 2021 (88). Isso é um dado interessante para o relatório: não indica necessariamente que menos países investem em P&D em 2021 — indica que menos países reportaram esse dado em 2021

##Diagnóstico: Pesquisadores em P&D

In [61]:
diagnostico_completo(df_pesquisadores)

== Tipos e nulos ==


,Tipo,Nulos (%),Valores únicos
Country Name,object,0.00,217
Country Code,object,0.00,217
Ano,int64,0.00,5
Valor,float64,67.10,357
Indicador,object,0.00,1



== Estatísticas descritivas ==


,Valor
count,357.00
mean,2209.66
std,2105.93
min,5.94
25%,454.80
50%,1594.22
75%,3558.20
max,9071.45



== Cobertura por ano (nº de países com dado) ==


,Valor
Ano,
2001,56
2006,64
2011,76
2016,78
2021,83



== Duplicatas país+ano ==
0


### Observações — Pesquisadores em P&D (por milhão hab.)

- **Cobertura:** 357 de 1085 combinações possíveis têm dado (67,1% de nulos) — cobertura ainda pior que Gasto em P&D (59,63%)
- **Média:** 2.209,7 | **Mediana:** 1.594,2 — mediana menor que a média, distribuição assimétrica (poucos países com número muito alto de pesquisadores puxam a média para cima)
- **Dispersão (desvio padrão):** 2.105,9 — quase do tamanho da própria média, dados muito dispersos entre países
- **Mínimo / Máximo:** 5,9 / 9.071,5 pesquisadores por milhão de hab. — intervalo enorme, mas plausível (países com pouca estrutura de pesquisa vs. países como Coreia do Sul/Israel com forte base científica)
- **Cobertura por ano:** cresce de forma constante — 56 (2001) → 64 (2006) → 76 (2011) → 78 (2016) → 83 (2021). Diferente do Gasto em P&D (que caiu no fim), aqui a tendência é de melhora contínua na disponibilidade do dado ao longo do tempo
- **Duplicatas:** 0 — melt correto

**Ponto de atenção:** a cobertura em 2001 (56 países) é bem menor que em 2021 (83) — quase 50% a mais de países reportando no fim do período. Isso é ainda mais relevante que no indicador de P&D, porque compromete comparações do Top 10 entre os anos mais antigos e recentes: o "pool" de concorrência em 2001 é bem menor.

##Diagnóstico: Pedidos de patentes

In [62]:
diagnostico_completo(df_patentes_resd)

== Tipos e nulos ==


,Tipo,Nulos (%),Valores únicos
Country Name,object,0.00,217
Country Code,object,0.00,217
Ano,int64,0.00,5
Valor,float64,51.61,374
Indicador,object,0.00,1



== Estatísticas descritivas ==


,Valor
count,525.00
mean,14369.82
std,92219.91
min,1.00
25%,32.00
50%,212.00
75%,1300.00
max,1426644.00



== Cobertura por ano (nº de países com dado) ==


,Valor
Ano,
2001,88
2006,96
2011,106
2016,122
2021,113



== Duplicatas país+ano ==
0


### Observações — Pedidos de patentes

- **Cobertura:** 525 de 1085 combinações possíveis têm dado (48,39% de nulos) — a melhor cobertura entre os 3 indicadores vistos até agora

- **Média:** 14.369,8 | **Mediana:** 212 — diferença enorme entre média e mediana, sinal de distribuição extremamente assimétrica: uns poucos países (provavelmente China, EUA, Japão) registram centenas de milhares de pedidos, enquanto a maioria fica na casa das centenas

- **Dispersão (desvio padrão):** 92.219,9 — muito maior que a própria média, confirma a forte concentração em poucos países

- **Mínimo / Máximo:** 1 / 1.426.644 — intervalo gigantesco (de 1 único pedido a mais de 1,4 milhão), plausível dado que esse indicador é contagem absoluta (não % nem por-milhão-de-habitante como os anteriores), então países grandes dominam naturalmente

- **Cobertura por ano:** cresce de 88 (2001) para 122 (2016), cai um pouco para 113 (2021) — mesma ressalva de cobertura desigual entre anos-base
- **Duplicatas:** 0 — melt correto



**Ponto de atenção para consolidação:** este indicador de patentes é **absoluto** (contagem bruta), enquanto Gasto em P&D e Pesquisadores são **relativos** (% ou por milhão de habitantes). Isso significa que comparar diretamente "quem tem mais patentes" favorece países grandes (população/economia), diferente dos outros dois indicadores, que já são normalizados. Vale mencionar essa diferença de natureza entre indicadores na Fase 2, ao interpretar correlações.

##Diagnóstico: Exportações de alta tecnologia

In [63]:
diagnostico_completo(df_exportacoes)

== Tipos e nulos ==


,Tipo,Nulos (%),Valores únicos
Country Name,object,0.00,217
Country Code,object,0.00,217
Ano,int64,0.00,5
Valor,float64,56.22,470
Indicador,object,0.00,1



== Estatísticas descritivas ==


,Valor
count,475.00
mean,10.31
std,12.22
min,0.00
25%,1.72
50%,6.43
75%,15.33
max,70.55



== Cobertura por ano (nº de países com dado) ==


,Valor
Ano,
2001,0
2006,0
2011,143
2016,166
2021,166



== Duplicatas país+ano ==
0


###  Observações — Exportações de alta tecnologia (%)

- **Cobertura:** 475 de 1085 combinações possíveis têm dado (56,22% de nulos)

- **Média:** 10,31% | **Mediana:** 6,43% — mediana bem menor que a média, distribuição assimétrica (poucos países exportadores de alta tecnologia muito fortes puxam a média para cima)

- **Dispersão (desvio padrão):** 12,22 — maior que a própria média, dados bastante dispersos

- **Mínimo / Máximo:** 0% / 70,55% — intervalo plausível (0% para países sem exportação de tecnologia, até um país fortemente especializado em tech)

- **Duplicatas:** 0 — melt correto

**Problema real de cobertura:** diferente dos outros indicadores (que tinham cobertura desigual mas presente em todos os anos), este indicador tem **ZERO países com dado em 2001 e 2006**. Cobertura só começa a existir a partir de 2011 (143 países), 2016 (166) e 2021 (166).

**O que isso significa na prática:** não dá para montar um Top 10 de Exportações de alta tecnologia para os anos 2001 e 2006 — simplesmente não existe dado nesse indicador para essas datas na base do Banco Mundial. Isso não é erro do nosso código (o `carregar_indicador_bm` e o filtro de anos estão corretos) — é uma limitação real da fonte de dados: esse indicador provavelmente começou a ser sistematicamente coletado/publicado só a partir dos anos 2010.

**Ação necessária:** avaliar se, para este indicador específico, o grupo apresenta o Top 10 apenas para 2011/2016/2021 (com uma nota explicando a ausência de dados anteriores), ou se buscamos uma fonte alternativa para os anos faltantes. Vale confirmar isso com o professor/guia do trabalho, já que os outros 4 indicadores têm os 5 anos completos.

##Carregamento e limpeza: Patentes, Marcas e Desenhos Industriais (WIPO)

In [64]:
CAMINHO_WIPO = CAMINHO + "Indicadores_PeD/patents_trademarks_industrial_design.csv"

df_patentes_wipo = carregar_indicador_wipo(
    CAMINHO_WIPO,
    "1.1 - Patent - Total patent applications",
    "Pedidos de patentes (WIPO)",
    anos_wipo
)

df_marcas_wipo = carregar_indicador_wipo(
    CAMINHO_WIPO,
    "2.1 - Trademark - Total classes in trademark applications",
    "Pedidos de marcas",
    anos_wipo
)

df_desenhos_wipo = carregar_indicador_wipo(
    CAMINHO_WIPO,
    "3.1 - Industrial design - Total designs in applications",
    "Pedidos de desenhos industriais",
    anos_wipo
)

for nome, d in [("Patentes WIPO", df_patentes_wipo), ("Marcas", df_marcas_wipo),
                ("Desenhos industriais", df_desenhos_wipo)]:
    print(nome, "->", d.shape, "| países únicos:", d["Country Name"].nunique())

Patentes WIPO -> (880, 5) | países únicos: 176
Marcas -> (915, 5) | países únicos: 183
Desenhos industriais -> (815, 5) | países únicos: 163


## Diagnóstico: Pedidos de patentes (WIPO)

In [65]:
diagnostico_completo(df_patentes_wipo)

== Tipos e nulos ==


,Tipo,Nulos (%),Valores únicos
Country Name,object,0.00,176
Country Code,object,0.57,175
Ano,int64,0.00,5
Valor,float64,27.84,430
Indicador,object,0.00,1



== Estatísticas descritivas ==


,Valor
count,635.00
mean,20040.75
std,112429.49
min,1.00
25%,27.00
50%,250.00
75%,2037.00
max,1796738.00



== Cobertura por ano (nº de países com dado) ==


,Valor
Ano,
2004,113
2009,100
2014,139
2019,137
2024,146



== Duplicatas país+ano ==
0


###  Observações — Pedidos de patentes (WIPO)

- **Cobertura:** 635 de 880 combinações possíveis têm dado (27,84% de nulos) — melhor cobertura entre todos os indicadores vistos até agora (mesmo indicador do Banco Mundial tinha 48,39% de nulos)

- **Média:** 20.040,8 | **Mediana:** 250 — diferença extrema entre média e mediana, distribuição fortemente assimétrica (poucos países concentram a maior parte dos pedidos — provavelmente China/EUA)

- **Dispersão (desvio padrão):** 112.429,5 — mais de 5x a média, confirma concentração extrema em poucos países

- **Mínimo / Máximo:** 1 / 1.796.738 — intervalo ainda maior que o mesmo indicador no Banco Mundial (1.426.644), plausível pois são fontes/metodologias diferentes de contagem

- **Cobertura por ano:** oscila entre 100-146 países, sem tendência clara de queda ou crescimento constante — 113 (2004) → 100 (2009) → 139 (2014) → 137 (2019) → 146 (2024)

- **Duplicatas:** 0 — melt correto

**Nulos em `Country Code`:** diferente do Banco Mundial (nunca tinha nulo em Country Code), aqui 0,57% das linhas têm código de país ausente. Precisa investigar: pode ser um "Country Name" sem correspondência de código no arquivo original da WIPO (ex: território ou nome incomum). Vale rodar uma checagem antes de seguir:



In [66]:
df_patentes_wipo[df_patentes_wipo["Country Code"].isna()]["Country Name"].unique()

array(['Namibia'], dtype=object)

###  Nota — Namíbia com Country Code nulo (não corrigido)

Identificado que o código "NA" da Namíbia é lido como nulo pelo pandas. Não afeta o
escopo deste trabalho, já que o objetivo é somente o Top 10 de cada indicador — caso apareça no top 10, será preciso tratar.


###Diagnóstico: Pedidos de marcas

In [67]:
diagnostico_completo(df_marcas_wipo)

== Tipos e nulos ==


,Tipo,Nulos (%),Valores únicos
Country Name,object,0.00,183
Country Code,object,0.55,182
Ano,int64,0.00,5
Valor,float64,33.77,583
Indicador,object,0.00,1



== Estatísticas descritivas ==


,Valor
count,606.00
mean,71869.05
std,456370.77
min,1.00
25%,929.25
50%,6436.50
75%,29589.25
max,7904365.00



== Cobertura por ano (nº de países com dado) ==


,Valor
Ano,
2004,90
2009,79
2014,134
2019,148
2024,155



== Duplicatas país+ano ==
0


###  Observações — Pedidos de marcas

- **Cobertura:** 606 de 915 combinações possíveis têm dado (33,77% de nulos) — segunda melhor cobertura entre todos os indicadores vistos

- **Média:** 71.869 | **Mediana:** 6.436,5 — diferença extrema entre média e mediana, distribuição fortemente assimétrica (concentração muito forte em poucos países, provavelmente China, com volume de marcas registradas ordens de grandeza acima do resto)

- **Dispersão (desvio padrão):** 456.370,8 — mais de 6x a média, a maior dispersão relativa vista até agora entre todos os indicadores

- **Mínimo / Máximo:** 1 / 7.904.365 — intervalo extremo, mas plausível dado o boom de registro de marcas na China nos últimos anos

- **Cobertura por ano:** cresce de forma quase constante — 90 (2004) → 79 (2009, única queda) → 134 (2014) → 148 (2019) → 155 (2024)

- **Duplicatas:** 0 — melt correto

- **Nulo em Country Code:** mesmo caso da Namíbia (0,55%) — não corrigido, conforme decisão de priorização já registrada

**Nota:** este é o indicador com maior concentração observada até agora (desvio padrão 6x a média). Reforça que o Top 10 de marcas provavelmente será dominado por 1-2 países com folga muito grande sobre o resto..

###Diagnóstico: Pedidos de desenhos industriais

In [68]:
diagnostico_completo(df_desenhos_wipo)

== Tipos e nulos ==


,Tipo,Nulos (%),Valores únicos
Country Name,object,0.00,163
Country Code,object,0.61,162
Ano,int64,0.00,5
Valor,float64,36.56,406
Indicador,object,0.00,1



== Estatísticas descritivas ==


,Valor
count,517.00
mean,9971.02
std,59716.03
min,1.00
25%,55.00
50%,335.00
75%,2267.00
max,907420.00



== Cobertura por ano (nº de países com dado) ==


,Valor
Ano,
2004,64
2009,72
2014,117
2019,126
2024,138



== Duplicatas país+ano ==
0


### Observações — Pedidos de desenhos industriais

- **Cobertura:** 517 de 815 combinações possíveis têm dado (36,56% de nulos)

- **Média:** 9.971 | **Mediana:** 335 — diferença grande entre média e mediana, distribuição assimétrica (concentração em poucos países, mesmo padrão dos outros indicadores WIPO)

- **Dispersão (desvio padrão):** 59.716 — cerca de 6x a média, concentração forte, mas um pouco menos extrema que Marcas

- **Mínimo / Máximo:** 1 / 907.420 — intervalo grande, plausível dado o padrão já visto nos outros indicadores WIPO

- **Cobertura por ano:** cresce de forma constante — 64 (2004) → 72 (2009) → 117 (2014) → 126 (2019) → 138 (2024). Esse é o indicador com o crescimento de cobertura mais acentuado (mais que dobrou do início ao fim do período)

- **Duplicatas:** 0 — melt correto
- **Nulo em Country Code:** mesmo caso da Namíbia (0,61%) — não corrigido, mesma decisão de priorização

## 3.1 Padronização de nomes de país

Antes de consolidar, precisamos alinhar os nomes de país entre Banco Mundial e WIPO.
Nesta etapa, identificamos de forma sistemática todos os nomes que não batem entre
as duas fontes, para depois construir um dicionário de correspondência.

###Comparar as duas listas de nomes de país

In [69]:
# Conjunto de nomes de país em cada fonte (usamos set() para poder comparar)
paises_bm = set(df_pd_pib["Country Name"].unique())
paises_wipo = set(df_patentes_wipo["Country Name"].unique())

# Nomes que existem no Banco Mundial mas não na WIPO (com esse texto exato)
so_no_bm = sorted(paises_bm - paises_wipo)

# Nomes que existem na WIPO mas não no Banco Mundial (com esse texto exato)
so_na_wipo = sorted(paises_wipo - paises_bm)

print(f"Nomes só no Banco Mundial ({len(so_no_bm)}):")
print(so_no_bm)

print(f"\nNomes só na WIPO ({len(so_na_wipo)}):")
print(so_na_wipo)

Nomes só no Banco Mundial (67):
['Afghanistan', 'American Samoa', 'Aruba', 'Bahamas, The', 'Bermuda', 'Bolivia', 'British Virgin Islands', 'Cayman Islands', 'Channel Islands', 'Congo, Dem. Rep.', 'Congo, Rep.', "Cote d'Ivoire", 'Curacao', 'Czechia', 'Egypt, Arab Rep.', 'Equatorial Guinea', 'Eritrea', 'Faroe Islands', 'Fiji', 'French Polynesia', 'Gambia, The', 'Gibraltar', 'Greenland', 'Guam', 'Hong Kong SAR, China', 'Iran, Islamic Rep.', 'Isle of Man', 'Kiribati', "Korea, Dem. People's Rep.", 'Korea, Rep.', 'Kosovo', 'Kyrgyz Republic', 'Lao PDR', 'Libya', 'Macao SAR, China', 'Maldives', 'Marshall Islands', 'Micronesia, Fed. Sts.', 'Moldova', 'Myanmar', 'Nauru', 'Netherlands', 'New Caledonia', 'Northern Mariana Islands', 'Palau', 'Puerto Rico (US)', 'Sint Maarten (Dutch part)', 'Slovak Republic', 'Solomon Islands', 'Somalia, Fed. Rep.', 'South Sudan', 'St. Kitts and Nevis', 'St. Lucia', 'St. Martin (French part)', 'St. Vincent and the Grenadines', 'Suriname', 'Tanzania', 'Timor-Leste', 

##Dicionário de correspondência construído

Das 26 divergências de nome na WIPO, todas têm correspondência direta com nomes do
Banco Mundial (apenas grafia/convenção diferente). Os demais 41 nomes exclusivos do
Banco Mundial são territórios/países pequenos sem presença na base WIPO (ex: West
Bank and Gaza, Kosovo, South Sudan, ilhas do Pacífico) — ficam de fora da junção com
WIPO por ausência real de dado, não por erro de nomenclatura.

In [70]:
correspondencia_paises = {
    "Bahamas": "Bahamas, The",
    "Bolivia (Plurinational State of)": "Bolivia",
    "China, Hong Kong SAR": "Hong Kong SAR, China",
    "China, Macao SAR": "Macao SAR, China",
    "Congo": "Congo, Rep.",
    "Czech Republic": "Czechia",
    "Côte d'Ivoire": "Cote d'Ivoire",
    "Democratic People's Republic of Korea": "Korea, Dem. People's Rep.",
    "Democratic Republic of the Congo": "Congo, Dem. Rep.",
    "Egypt": "Egypt, Arab Rep.",
    "Gambia": "Gambia, The",
    "Iran (Islamic Republic of)": "Iran, Islamic Rep.",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Lao People's Democratic Republic": "Lao PDR",
    "Netherlands (Kingdom of the)": "Netherlands",
    "Republic of Korea": "Korea, Rep.",
    "Republic of Moldova": "Moldova",
    "Saint Kitts and Nevis": "St. Kitts and Nevis",
    "Saint Lucia": "St. Lucia",
    "Saint Vincent and the Grenadines": "St. Vincent and the Grenadines",
    "Slovakia": "Slovak Republic",
    "Türkiye": "Turkiye",
    "United Republic of Tanzania": "Tanzania",
    "United States of America": "United States",
    "Venezuela (Bolivarian Republic of)": "Venezuela, RB",
    "Yemen": "Yemen, Rep.",
}

print(f"Total de correspondências mapeadas: {len(correspondencia_paises)}")

Total de correspondências mapeadas: 26


In [71]:
# ============================================================
# Padronizar nomes de país nas 3 tabelas WIPO, usando o dicionário
# ============================================================
for d in [df_patentes_wipo, df_marcas_wipo, df_desenhos_wipo]:
    d["Country Name"] = d["Country Name"].replace(correspondencia_paises)

# Confirma que a divergência diminuiu
paises_wipo_ajustado = set(df_patentes_wipo["Country Name"].unique())
print("Nomes ainda só na WIPO (deve ser 0):", sorted(paises_wipo_ajustado - paises_bm))

Nomes ainda só na WIPO (deve ser 0): []


Padronização concluída

Aplicado o dicionário de 26 correspondências nas 3 tabelas WIPO (Patentes, Marcas,
Desenhos Industriais). Confirmado: nenhum nome de país da WIPO ficou fora do padrão
do Banco Mundial após a substituição. As tabelas estão prontas para consolidação.

##3.2 Consolidação das bases

Com os nomes de país padronizados, empilhamos as 7 tabelas (4 do Banco Mundial + 3
da WIPO) em um único dataframe, todas já no mesmo formato long
(Country Name / Country Code / Ano / Valor / Indicador).

In [72]:
# ============================================================
# Consolidação: empilhar as 7 tabelas em um único dataframe
# ============================================================
df_consolidado = pd.concat([
    df_pd_pib,
    df_pesquisadores,
    df_patentes_resd,
    df_exportacoes,
    df_patentes_wipo,
    df_marcas_wipo,
    df_desenhos_wipo
], ignore_index=True)

df_consolidado.shape
df_consolidado["Indicador"].unique()
df_consolidado["Indicador"].value_counts()

,count
Indicador,
Gasto em P&D (% do PIB),1085
Pesquisadores em P&D (por milhão hab.),1085
Pedidos de patentes,1085
Exportações de alta tecnologia (%),1085
Pedidos de marcas,915
Pedidos de patentes (WIPO),880
Pedidos de desenhos industriais,815


###Consolidação confirmada

Dataframe único com os 7 indicadores empilhados. Contagem de linhas por indicador
bate com o esperado: 1085 para cada indicador do Banco Mundial (217 países × 5 anos),
e 915/880/815 para os indicadores WIPO (variação por ausência de registro de
determinado tipo de propriedade intelectual em alguns países).

#4. Modelagem (Top 10)

In [73]:
top10_pd_pib = top10_por_indicador_ano(df_consolidado, "Gasto em P&D (% do PIB)", 2021)
top10_pd_pib

,Country Name,Valor
0,Israel,5.76
1,"Korea, Rep.",4.60
2,United States,3.47
3,Sweden,3.42
4,Belgium,3.41
5,Japan,3.27
6,Austria,3.26
7,Switzerland,3.25
8,Germany,3.08
9,Finland,3.01


### Top 10 — Gasto em P&D (% do PIB), 2021

Israel lidera com folga (5,76%), seguido por Coreia do Sul (4,60%). O ranking é
dominado por economias desenvolvidas da Europa e Ásia — nenhum país em desenvolvimento
aparece. Os valores batem com o máximo (5,76%) identificado no diagnóstico estatístico,
confirmando consistência entre as etapas.

In [74]:
top10_pesquisadores = top10_por_indicador_ano(df_consolidado, "Pesquisadores em P&D (por milhão hab.)", 2021)
top10_patentes_bm = top10_por_indicador_ano(df_consolidado, "Pedidos de patentes", 2021)
top10_exportacoes = top10_por_indicador_ano(df_consolidado, "Exportações de alta tecnologia (%)", 2021)

top10_patentes_wipo = top10_por_indicador_ano(df_consolidado, "Pedidos de patentes (WIPO)", 2024)
top10_marcas = top10_por_indicador_ano(df_consolidado, "Pedidos de marcas", 2024)
top10_desenhos = top10_por_indicador_ano(df_consolidado, "Pedidos de desenhos industriais", 2024)



In [75]:
top10_pesquisadores

,Country Name,Valor
0,"Korea, Rep.",9071.45
1,Sweden,8159.80
2,Singapore,7956.40
3,Finland,7870.43
4,Denmark,7708.24
5,Norway,7228.72
6,Iceland,6941.30
7,Austria,6323.00
8,Netherlands,6003.87
9,Switzerland,6003.41


In [76]:
top10_patentes_bm

,Country Name,Valor
0,China,1426644.00
1,United States,262244.00
2,Japan,222452.00
3,"Korea, Rep.",186245.00
4,Germany,39822.00
5,India,26267.00
6,Russian Federation,19569.00
7,France,13386.00
8,United Kingdom,11592.00
9,Italy,10281.00


In [77]:
top10_exportacoes

,Country Name,Valor
0,"Hong Kong SAR, China",70.55
1,Cuba,70.09
2,Cayman Islands,67.77
3,Philippines,64.23
4,Singapore,54.97
5,Malaysia,51.68
6,Viet Nam,41.54
7,Papua New Guinea,40.05
8,"Korea, Rep.",36.01
9,Iceland,33.49


In [78]:
top10_patentes_wipo

,Country Name,Valor
0,China,1796738.00
1,United States,503283.00
2,Japan,420991.00
3,"Korea, Rep.",296041.00
4,Germany,133790.00
5,India,76473.00
6,France,51880.00
7,United Kingdom,46840.00
8,Switzerland,41341.00
9,Netherlands,26445.00


In [79]:
top10_marcas

,Country Name,Valor
0,China,7314677.00
1,United States,840047.00
2,Russian Federation,559679.00
3,India,533417.00
4,Brazil,436355.00
5,Germany,434215.00
6,Turkiye,401096.00
7,France,363218.00
8,United Kingdom,348712.00
9,Japan,340721.00


In [80]:
top10_desenhos

,Country Name,Valor
0,China,907420.00
1,Germany,70262.00
2,United States,67095.00
3,Italy,63709.00
4,"Korea, Rep.",60175.00
5,Turkiye,44021.00
6,France,42007.00
7,United Kingdom,41370.00
8,India,39116.00
9,Japan,33536.00
